In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as ani
import itertools
from scipy.linalg import norm
plt.rc('animation', html='jshtml')

np.set_printoptions(precision=3)

In [6]:
dt = 0.005 # tidsstep

number_of_particles = 2
#startdata
x0s = np.random.rand(number_of_particles)
y0s = np.random.rand(number_of_particles)
vx0s = np.cos(np.random.rand(number_of_particles)*2*np.pi)
vy0s = np.sin(np.random.rand(number_of_particles)*2*np.pi)

#antal partikler i lister(startdata)
range_of_particles  = range(number_of_particles)

R = 0.01 #radius partikler

#plot
fig, ax = plt.subplots()
partikler, = ax.plot([], [], 'ro')

box = ax.plot([0,1,1,0,0], [0,0,1,1,0], 'k')

#klasse
class particles:
    def __init__(self, xs, ys, vys, vxs, r):
        self.x0s = xs
        self.y0s = ys
        self.vx0s = vxs
        self.vy0s = vys
        
        self.n_particles = len(xs)
        self.radius = r
        
    def velocity_propagation(self):
        for particle in range_of_particles:
            self.x0s[particle] += self.vx0s[particle]*dt
            self.y0s[particle] += self.vy0s[particle]*dt

    def collision_with_walls(self):
        for particle in range_of_particles:

                #højre side af kassen
            if self.x0s[particle] >= 1:
                self.vx0s[particle] *= -1

                #venstre side af kassen 
            if self.x0s[particle] <= 0:
                self.vx0s[particle] *= -1

                #toppen af kassen
            if self.y0s[particle] >= 1:
                self.vy0s[particle] *= -1

                #bunden af kassen
            if self.y0s[particle] <= 0:
                self.vy0s[particle] *= -1
    
        #partikler.set_data(self.x0s, self.y0s)
    
    def particle_particle_collision(self):
        
        for i,j in itertools.combinations(range(0,self.n_particles),r=2):
            
            d = np.sqrt((self.x0s[i]-self.x0s[j])**2 + (self.y0s[i]-self.y0s[j])**2)
            
            if d<=2*self.radius:
                # detected collision
                
                # handle collision event. 
                self._handle_particle_collision_event(i,j)
    
    def _handle_particle_collision_event(self,i,j):
        
        #før kollision
        
        #parallelle positions vektorkoordinater
        rx_parallel = self.x0s[i]-self.x0s[j]
        ry_parallel = self.y0s[i]-self.y0s[j]
        
        
        #vektorer
        r_parallel = np.array([rx_parallel, ry_parallel])
        
        rhat_parallel = 1/np.linalg.norm(r_parallel)*r_parallel
        
        #vinkelret vektor
        rhat_vinkel =1/np.linalg.norm(r_parallel)*np.array([-ry_parallel, rx_parallel])
        
        #hastigheder før kollision
        p1_vx = self.vx0s[i]
        p1_vy = self.vy0s[i]
        p1_v = np.array([p1_vx, p1_vy])
        
        p2_vx = self.vx0s[j]
        p2_vy = self.vy0s[j]
        p2_v = np.array([p2_vx, p2_vy])
        
        
        p1_v_parallel = np.dot(p1_v, rhat_parallel)
        p1_v_vinkel = np.dot(p1_v, rhat_vinkel)
        
        p2_v_parallel = np.dot(p2_v, rhat_parallel)
        p2_v_vinkel = np.dot(p2_v, rhat_vinkel)
        
        #efterkollision
        slutv1 = p2_v_parallel*rhat_parallel+p1_v_vinkel*rhat_vinkel
        slutv2 = p1_v_parallel*rhat_parallel+p2_v_vinkel*rhat_vinkel

        #vektorkomposenter       
        self.vx0s[i] = slutv1[0]
        self.vy0s[i] = slutv1[1]
        self.vx0s[j] = slutv2[0]
        self.vy0s[j] = slutv2[1]
    
par = particles(x0s, y0s, vx0s, vy0s, R)
plt.close()

In [3]:
#update funktion
def update(i):

    par.velocity_propagation()
    par.collision_with_walls()
    par.particle_particle_collision()
    
    partikler.set_data(par.x0s, par.y0s)
    return partikler,

plt.close()

anim=ani.FuncAnimation(fig, update, frames=400, interval=10, blit=True)

In [4]:
anim

In [5]:
# writer = ani.writers['pillow'](fps=50)
# anim.save('kollosion.gif', writer=writer) 